In [ ]:
!pip3 install optimum
!pip install -U flash-attn --no-build-isolation
!pip install transformers==4.43.3
!pip install pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.1/424.1 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 20.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 re

In [ ]:
from IPython.display import Audio
import IPython.display as ipd
from tqdm import tqdm
from transformers import BarkModel, AutoProcessor, AutoTokenizer
import torch
import json
import numpy as np
import pickle
import io
from scipy.io import wavfile
from pydub import AudioSegment
import ast

In [ ]:
#Load the generated .pkl file - This is defined as data.pkl, while files will be generated as "section#.pkl" instead. In this prototype setup we sort to rename the section files into data.pkl and run through each at a time.
with open('data.pkl', 'rb') as file:
    PODCAST_TEXT = pickle.load(file)

FileNotFoundError: [Errno 2] No such file or directory: 'data.pkl'

In [ ]:
#we setup our bark TTS model and settings
bark_processor = AutoProcessor.from_pretrained("suno/bark")
bark_model = BarkModel.from_pretrained("suno/bark", torch_dtype=torch.float16).to("cuda")
bark_sampling_rate = 24000

tokenizer_config.json:   0%|          | 0.00/353 [00:00<?, ?B/s]

speaker_embeddings_path.json:   0%|          | 0.00/61.1k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/8.81k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/4.49G [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/usr/local/lib/python3.10/dist-packages/transformers/models/encodec/modeling_encodec.py:120: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer("padding_total", torch.tensor(kernel_size - stride, dtype=torch.int64), persistent=False)


generation_config.json:   0%|          | 0.00/4.91k [00:00<?, ?B/s]

In [ ]:
#containers
generated_segments = []
sampling_rates = []

In [ ]:
#use our single GPU
device="cuda"

Function generate text for speaker 1

In [ ]:
#Here we define what speakers to use and settings. We use Speaker9 and speaker6 as speaker6 is barks best model and speaker9 is barks only female voice for the english language.
def generate_speaker_audio(text, voice_preset, temperature=0.9, semantic_temperature=0.8):
    """Generate audio using Bark for a specific speaker with distinct voice preset"""
    inputs = bark_processor(text, voice_preset=voice_preset).to(device)
    speech_output = bark_model.generate(**inputs, temperature=temperature, semantic_temperature=semantic_temperature)
    audio_arr = speech_output[0].cpu().numpy()
    return audio_arr, bark_sampling_rate


def generate_speaker1_audio(text):
    """Generate audio for Speaker 1 using Bark"""
    return generate_speaker_audio(text, voice_preset="v2/en_speaker_9")


def generate_speaker2_audio(text):
    """Generate audio for Speaker 2 using Bark"""
    return generate_speaker_audio(text, voice_preset="v2/en_speaker_6")


In [ ]:
#We define a function to convert the array of the generated script into sound
def numpy_to_audio_segment(audio_arr, sampling_rate):
    """Convert numpy array to AudioSegment"""
    audio_int16 = (audio_arr * 32767).astype(np.int16)

    byte_io = io.BytesIO()
    wavfile.write(byte_io, sampling_rate, audio_int16)
    byte_io.seek(0)

    return AudioSegment.from_wav(byte_io)

In [ ]:
#Inspect
PODCAST_TEXT

NameError: name 'PODCAST_TEXT' is not defined

In [ ]:
#We load the pickle file as a tuple
ast.literal_eval(PODCAST_TEXT)

[('Speaker 1',
  'To understand disruptive innovation, we need to go back to the origins of the concept. The authors of this article searched the Web of Science database to identify broad patterns in early formulations of disruption theory.'),
 ('Speaker 2',
  "Hmm, I've never really used that database before. Can you explain how they used it?"),
 ('Speaker 1',
  'They searched for articles citing key papers by Bower and Christensen, and then examined the usage of disruption theory terminology in academic sources between 1993 and 2016.'),
 ('Speaker 2',
  "Umm, that's a lot of data to sift through. Did they get any input from experts in the field?"),
 ('Speaker 1',
  'Yes, they solicited input from experts who proposed relevant books and articles that would be challenging to identify through their methodology.'),
 ('Speaker 2',
  'Ah, that makes sense. I can imagine that would be helpful. But what kind of questions did they ask the experts?'),
 ('Speaker 1',
  'They asked for recommend

In [ ]:
#Generate audio using definitions of speakers and array to audio
final_audio = None

for speaker, text in tqdm(ast.literal_eval(PODCAST_TEXT), desc="Generating podcast segments", unit="segment"):
    if speaker == "Speaker 1":
        audio_arr, rate = generate_speaker1_audio(text)
    else:
        audio_arr, rate = generate_speaker2_audio(text)

    audio_segment = numpy_to_audio_segment(audio_arr, rate)

    if final_audio is None:
        final_audio = audio_segment
    else:
        final_audio += audio_segment

Generating podcast segments:   0%|          | 0/10 [00:00<?, ?segment/s]

en_speaker_9_semantic_prompt.npy:   0%|          | 0.00/3.06k [00:00<?, ?B/s]

en_speaker_9_coarse_prompt.npy:   0%|          | 0.00/8.94k [00:00<?, ?B/s]

en_speaker_9_fine_prompt.npy:   0%|          | 0.00/17.8k [00:00<?, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token.As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Generating podcast segments:  10%|█         | 1/10 [00:59<08:57, 59.77s/segment]

en_speaker_6_semantic_prompt.npy:   0%|          | 0.00/2.60k [00:00<?, ?B/s]

en_speaker_6_coarse_prompt.npy:   0%|          | 0.00/7.55k [00:00<?, ?B/s]

en_speaker_6_fine_prompt.npy:   0%|          | 0.00/15.0k [00:00<?, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
Generating podcast segments:  20%|██        | 2/10 [01:21<04:57, 37.14s/segment]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
Generating podcast segments:  30%|███       | 3/10 [02:17<05:22, 46.06s/segment]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
Generating podcast segments:  40%|████      | 4/10 [03:03<04:36, 46.00s/segment]The att

In [ ]:
#Save as mp3
final_audio.export("_podcast.mp3",
                  format="mp3",
                  bitrate="192k",
                  parameters=["-q:a", "0"])

<_io.BufferedRandom name='_podcast.mp3'>